# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [9]:
!python3 -m pip install -qU jq lark langchain langchain_elasticsearch langchain_openai tiktoken

In [11]:
import sys
print("Kernel:", sys.executable)
!{sys.executable} -m pip install -U "langchain>=0.2.12" "langchain-openai>=0.1.16" "langchain-elasticsearch>=0.2.0" "elasticsearch>=8.12.0" "tiktoken"

Kernel: c:\Users\Lain\anaconda3\python.exe
  Using cached langchain_elasticsearch-0.3.2-py3-none-any.whl.metadata (8.3 kB)
  Using cached tiktoken-0.11.0-cp312-cp312-win_amd64.whl.metadata (6.9 kB)
  Using cached elasticsearch-8.19.0-py3-none-any.whl.metadata (9.2 kB)
  Using cached elastic_transport-8.17.1-py3-none-any.whl.metadata (3.8 kB)
Using cached langchain_elasticsearch-0.3.2-py3-none-any.whl (45 kB)
Using cached elasticsearch-8.19.0-py3-none-any.whl (926 kB)
Using cached tiktoken-0.11.0-cp312-cp312-win_amd64.whl (884 kB)
Using cached elastic_transport-8.17.1-py3-none-any.whl (64 kB)
  Attempting uninstall: tiktoken
    Found existing installation: tiktoken 0.9.0
    Uninstalling tiktoken-0.9.0:
      Successfully uninstalled tiktoken-0.9.0


  You can safely remove it manually.


In [4]:
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai import OpenAIEmbeddings
print("OK imports")

OK imports


In [8]:

from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial. 

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [11]:
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id
ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")

# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name="ironhack", #give it a meaningful name,
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [12]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [14]:
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [18]:
pip install jq


Note: you may need to restart the kernel to use updated packages.


In [19]:
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import jq
def metadata_func(record: dict, metadata: dict) -> dict:
    #Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    metadata["name"] = record.get("name", "")
    metadata["summary"] = record.get("summary", "")
    metadata["url"] = record.get("url", "")
    metadata["category"] = record.get("category", "")
    metadata["updated_at"] = record.get("updated_at", "")

    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800, chunk_overlap=400 #define chunk size and chunk overlap
)
docs = loader.load_and_split(text_splitter=text_splitter)

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [ ]:
documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name="ironhack",
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

llm = OpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

BulkIndexError: 10 document(s) failed to index.

In [31]:
llm = OpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)
retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

In [24]:
count = vectorstore.client.count(index="ironhack")["count"]
print("Docs en vs:", count)
print("Docs en memoria (len(docs)):", len(docs))

Docs en vs: 15
Docs en memoria (len(docs)): 15


# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [32]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide information on the sales team at NASA?', '2. How does the sales team at NASA operate?', '3. What are the responsibilities of the sales team at NASA?']


---- Answer ----
I'm sorry, I cannot answer that question as it is not mentioned in the retrieved context. The context only discusses the sales strategy and objectives for a tech company, not NASA or their sales team.


**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_core.prompts import PromptTemplate
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough

base_ret = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 8, "lambda_mult": 0.5})

mqr_prompt = PromptTemplate.from_template(
    "Generate {n} short, diverse search queries (one per line) for: {question}\nQueries:\n"
).partial(n=6)

mq = MultiQueryRetriever.from_llm(retriever=base_ret, llm=llm, prompt=mqr_prompt)

_context = RunnableParallel(
    context=mq | _combine_documents,
    question=RunnablePassthrough(),
)
chain_A = _context | LLM_CONTEXT_PROMPT | llm  

print(chain_A.invoke("Summarize the hybrid work policy and who approves PTO requests."))


INFO:langchain.retrievers.multi_query:Generated queries: ['1. "Hybrid work policy summary"', '2. "PTO request approval process"', '3. "Hybrid work policy decision maker"', '4. "Who can approve PTO requests?"', '5. "Hybrid work and time off guidelines"', '6. "PTO request authorization for hybrid work"']



The hybrid work policy requires employees to work from the office three days a week and allows for two days of remote work. PTO requests must be approved by the employee's supervisor and the HR department.


In [ ]:
policy_filter = {"term": {"metadata.category.keyword": "Policy"}} 

focused_ret = vectorstore.as_retriever(search_kwargs={"k": 6, "filter": policy_filter})
mq_focus = MultiQueryRetriever.from_llm(retriever=focused_ret, llm=llm)

_contextB = RunnableParallel(
    context=mq_focus | _combine_documents,
    question=RunnablePassthrough(),
)
chain_B = _contextB | LLM_CONTEXT_PROMPT | llm

print(chain_B.invoke("What is the PTO request process, including approvals and any blackout dates?"))


INFO:langchain.retrievers.multi_query:Generated queries: ['1. How does the PTO request process work, and what are the steps for getting approvals and dealing with blackout dates?', '2. Can you explain the process for requesting PTO, including how approvals are obtained and any blackout dates that may affect the request?', '3. What are the steps involved in the PTO request process, and how can one navigate through approvals and blackout dates?']


The PTO (Paid Time Off) request process is the procedure employees must follow in order to request time off from work. This process typically involves submitting a request to their supervisor or HR department, either through a designated form or through an online system. The request should include the dates and duration of the requested time off, as well as the reason for the request. 

Once the request is submitted, it will go through an approval process. This may involve multiple levels of approval, depending on the company's policies and procedures. The first level of approval is typically from the employee's direct supervisor, who will review the request and determine if it can be granted based on the employee's workload and the company's needs. If the supervisor approves the request, it may then go to a higher level of management for final approval. 

In some cases, there may be blackout dates for PTO requests. These are specific dates or periods of time during which employees are